# Rebuild Hippo time-course inputs and reproduce Figures 3–4

**Run All on Sherlock in the original `sc-dl` environment, within a high-memory compute allocation.** This notebook rebuilds both missing inputs, validates their historical cohort and matrix definitions, updates the active local path overrides, runs the main-figure preflight, and then executes **Figure 3 (b–d, f)** and **Figure 4 (c–e)** in fresh kernels using the same Python interpreter.

Review the settings cell once. The source directory must contain the eight processed `Hippo:eTN8_GEX_sub1` … `sub8` folders. Input files are read only. This starts from processed sublibrary H5ADs; quantification and cell calling must already have been performed.

## What is reconstructed

| File | Meaning | Historical check |
|---|---|---|
| `Hippo_timecourse_combined.h5ad` | Raw expression, original broad QC, all original layers and cell metadata | 212,891 cells × 62,757 features; all 54 sample counts |
| `Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad` | Fixed-reference top-specific-200 usage and CLR | 212,891 cells × 50 programs; 7,268 selected genes |

The later condition-specific QC is recorded as a flag in the projected object (**99,416 passing cells**); it does not remove rows from either reconstructed object. The figure helpers apply that flag and exclude sublibrary `s8` as in the recovered panel workflow.

The query comes from production `Hippo_timecourse.ipynb` cells 5, 14, 20, 23, and 27 (zero based); those source cells are also preserved verbatim in `recovery/raw_downloads/Hippo_timecourse.ipynb`. The selected projection comes from `recovery/raw_downloads/reproject_cnmf_compare_gene_sets.ipynb` cells 3, 5, 7, 9, 11, and 13. Scientific code is in `src/cc_perturbseq/hippo_rebuild.py` and `hippo_projection.py`.

See [the workflow notes](../../docs/HIPPO_REBUILD.md) for provenance, settings, dependencies, and restart behavior.


In [ ]:
# Repository-path bootstrap for local support modules.
from pathlib import Path
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing "
        "src/cc_perturbseq. Run this notebook from within the "
        "cc-perturbseq-manuscript-analysis checkout."
    )


REPO_ROOT = _find_repo_root()
for _support_path in (REPO_ROOT, REPO_ROOT / "src"):
    _support_path_text = str(_support_path)
    if _support_path_text not in sys.path:
        sys.path.insert(0, _support_path_text)

print(
    "Repository support paths added:",
    REPO_ROOT,
    "and",
    REPO_ROOT / "src",
)


## 1. Settings

Tracked defaults point to the documented Sherlock source directory and an Oak output directory. If the eight processed inputs were moved, change `INPUT_ROOT` below to their new parent directory. Existing `config/paths.local.yaml` or `CCPERTURBSEQ_PATHS` overrides are honored.

Keep `OVERWRITE = False` for normal use: a completed output is reused only when its recorded provenance and validation match. Set it to `True` to deliberately replace a previous reconstruction. Keep `STRICT_HISTORICAL = True` for manuscript reproduction; a cohort mismatch needs investigation.

By default the final stage executes both affected figure notebooks. Set `RUN_FIGURE_NOTEBOOKS = False` if this run should stop after reconstructing and reconnecting the inputs. Concatenation materializes the 2,118,711-cell pre-QC cohort; use a high-memory node. The earlier suggested 256 GB allocation is a starting estimate, not a measured guarantee.


In [ ]:
from datetime import datetime, timezone
import json
import tempfile
from IPython.display import display

from src.cc_perturbseq.figure_notebook import get_path, load_paths
from src.cc_perturbseq.hippo_workflow import (
    check_runtime, execute_figure_notebooks, run_main_preflight,
    validate_hippo_panel_inputs, write_hippo_overrides,
)

config_file, paths = load_paths(REPO_ROOT)
INPUT_ROOT = get_path(paths, "hippo_rebuild.input_root")
REBUILD_ROOT = get_path(paths, "hippo_rebuild.output_root")
SPECTRA_TPM = get_path(paths, "hippo_rebuild.gene_spectra_tpm")

OVERWRITE = False
STRICT_HISTORICAL = True
COMPRESSION = None  # Original uncompressed H5AD; optionally "gzip" or "lzf".
RUN_FIGURE_NOTEBOOKS = True
FIGURE_NOTEBOOKS = ("Figure_03.ipynb", "Figure_04.ipynb")

QUERY_H5AD = REBUILD_ROOT / "Hippo_timecourse_combined.h5ad"
PROJECTED_H5AD = (
    REBUILD_ROOT / "perturbseq_cNMF_projection" / "reprojection_gene_set_comparison"
    / "Hippo_timecourse_reprojected_cNMF_compare_gene_sets.cell_x_program.h5ad"
)
RUN_DIR = REBUILD_ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

display({
    "configuration": str(config_file), "input_root": str(INPUT_ROOT),
    "reference_spectra": str(SPECTRA_TPM), "query_output": str(QUERY_H5AD),
    "projection_output": str(PROJECTED_H5AD), "run_record": str(RUN_DIR),
    "execute_figures": RUN_FIGURE_NOTEBOOKS,
})


## 2. Check dependencies and direct inputs before loading the cohort

This lists every missing sublibrary explicitly. The projection reference must be the **gene-spectra TPM** file, not the similarly named consensus spectra used by program-annotation figures. Only the selected top-specific-200 mode is fitted; the HVG list and annotation workbook are not projection inputs. The workbook remains required by the main figure notebooks.


In [ ]:
versions = check_runtime(include_figures=RUN_FIGURE_NOTEBOOKS)
from src.cc_perturbseq.hippo_rebuild import (
    preflight_sublibraries, rebuild_combined, validate_combined,
)
from src.cc_perturbseq.hippo_projection import (
    project_top_specific_200, read_spectra, validate_projected,
)

input_records = preflight_sublibraries(INPUT_ROOT)
reference = read_spectra(SPECTRA_TPM, strict_historical=STRICT_HISTORICAL)
print("Reference shape:", reference.shape)
del reference
REBUILD_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=REBUILD_ROOT) as probe:
    probe.write(b"Hippo output write check")
RUN_DIR.mkdir(parents=True, exist_ok=False)
settings_record = {
    "input_files": input_records, "spectra_tpm": str(SPECTRA_TPM),
    "query_h5ad": str(QUERY_H5AD), "projected_h5ad": str(PROJECTED_H5AD),
    "strict_historical": STRICT_HISTORICAL, "overwrite": OVERWRITE,
    "compression": COMPRESSION, "versions": versions,
}
(RUN_DIR / "settings.json").write_text(json.dumps(settings_record, indent=2) + "\n")
display(input_records)
display(versions)


## 3. Recreate the expression query

The producer concatenates sublibraries with keys `s1`…`s8`, `index_unique="_"`, `join="outer"`, `merge="same"`, and `uns_merge="same"`. It retains cells with **1,200 < detected genes < 12,000**, **1,200 < total counts < 50,000**, **mitochondrial percentage < 10**, and biological sample unequal to `empty`, then applies the original 54-entry sample mapping. Fractional quantification values are preserved.

The output is written to a temporary sibling file, reopened and validated, and only then moved to the final path. Count layers and guide matrices are retained. This stage does not apply the later condition-specific mitochondrial/ribosomal filter.


In [ ]:
combined_report = rebuild_combined(
    INPUT_ROOT, QUERY_H5AD, overwrite=OVERWRITE,
    strict_historical=STRICT_HISTORICAL, compression=COMPRESSION,
)
(RUN_DIR / "combined_stage.json").write_text(json.dumps(combined_report, indent=2) + "\n")
print("Combined query:", combined_report["status"], QUERY_H5AD)
display(combined_report["validation"])


## 4. Recreate the final selected fixed-reference projection

For each of the 50 programs, genes are ranked by their fraction of total reference weight, then by program weight. The first-occurrence union of each program's top 200 genes is intersected with query symbols. Duplicate query symbols are summed, and expression is normalized using the original full-library totals to 1,000,000 per cell.

The existing spectra are fixed. The source multiplicative-update solve uses float32, Frobenius loss, 1,000 maximum iterations, tolerance `1e-5`, and seed `0`. CLR uses normalized usage, floor `1e-6`, renormalization, and centered log2 values. The named matrix consumed by the figures is `obsm["X_cnmf_clr_top_specific_200"]`, with explicit columns `P0`…`P49`.

Only this selected mode is stored. In this reconstructed file, `X` also holds selected normalized usage; downstream figures read the named CLR matrix. Iterations and any convergence warning are recorded; parameters are never changed automatically. The original selected solve reported 140 iterations, which is a comparison point rather than a required cross-version numerical identity.


In [ ]:
projection_report = project_top_specific_200(
    QUERY_H5AD, SPECTRA_TPM, PROJECTED_H5AD, overwrite=OVERWRITE,
    strict_historical=STRICT_HISTORICAL, compression=COMPRESSION,
)
(RUN_DIR / "projection_stage.json").write_text(json.dumps(projection_report, indent=2) + "\n")
display(projection_report["diagnostics"])
display(projection_report["validation"])


## 5. Validate the pair and reconnect the figure notebooks

Validation checks complete cell identity/order, metadata, the original usage-to-CLR calculation, program labels, and historical counts. The figures additionally require their marker genes and every requested condition after QC. Local configuration is updated only after pair validation; unrelated settings are preserved and the previous YAML is backed up under this run directory.


In [ ]:
pair_validation = validate_projected(
    QUERY_H5AD, PROJECTED_H5AD, strict_historical=STRICT_HISTORICAL,
)
panel_validation = validate_hippo_panel_inputs(QUERY_H5AD)
config_update = write_hippo_overrides(
    REPO_ROOT, QUERY_H5AD, PROJECTED_H5AD, backup_dir=RUN_DIR / "config_backups",
)
(RUN_DIR / "pair_validation.json").write_text(json.dumps(pair_validation, indent=2) + "\n")
(RUN_DIR / "panel_validation.json").write_text(json.dumps(panel_validation, indent=2) + "\n")
(RUN_DIR / "configuration_update.json").write_text(json.dumps(config_update, indent=2) + "\n")
display(pair_validation)
display(panel_validation)
display(config_update)


## 6. Preflight and execute Figures 3 and 4

The existing main-figure preflight checks all four main notebooks and their configured inputs. When it succeeds, the two affected notebooks run sequentially in clean kernels using **this Python environment**, with no execution time limit. Each notebook saves its normal figures/source tables; executed notebook copies, failure traces, a run receipt, and copies of successful figure directories are kept under the Oak run directory.

If a downstream notebook fails, the validated H5ADs remain available. Fix the reported error and rerun this final cell, or open the relevant main notebook directly. Successful execution does not by itself establish agreement with the assembled manuscript; inspect the generated figures and source tables.


In [ ]:
if RUN_FIGURE_NOTEBOOKS:
    preflight_report = run_main_preflight(REPO_ROOT, RUN_DIR / "main_figure_preflight.log")
    figure_report = execute_figure_notebooks(
        REPO_ROOT, RUN_DIR, notebook_names=FIGURE_NOTEBOOKS,
    )
    display(figure_report)
else:
    print("Both validated inputs are configured. Figure execution was disabled in settings.")

print("Expression query:", QUERY_H5AD)
print("Selected projection:", PROJECTED_H5AD)
print("Run records and completed figure copies:", RUN_DIR)
